In [1]:
import pandas as pd
import numpy as np

CITY = "Чернігів"
BASE_PRICE = 690
BASE_QTY = 4

df = pd.DataFrame([
    {"id": 1, "місто": CITY, "ціна": BASE_PRICE, "кількість": BASE_QTY},
    {"id": 2, "місто": f"  {CITY}  ", "ціна": f"{BASE_PRICE} грн", "кількість": np.nan},
    {"id": 3, "місто": CITY.upper(), "ціна": BASE_PRICE, "кількість": BASE_QTY + 1},
    {"id": 3, "місто": CITY.upper(), "ціна": BASE_PRICE, "кількість": BASE_QTY + 1},
    {"id": 5, "місто": CITY, "ціна": BASE_PRICE * 9, "кількість": BASE_QTY},
    {"id": 6, "місто": CITY, "ціна": f"{BASE_PRICE} грн", "кількість": np.nan},
    {"id": 7, "місто": CITY, "ціна": BASE_PRICE, "кількість": BASE_QTY - 1},
    {"id": 8, "місто": CITY, "ціна": BASE_PRICE, "кількість": BASE_QTY + 2}
])

print("--- Завдання 1: Початковий брудний набір ---")
print(df)

print("\n--- Завдання 2: Пропуски ---")
print("Кількість пропусків у кожному стовпці:")
print(df.isna().sum())

qty_median = df["кількість"].median()
df["кількість"] = df["кількість"].fillna(qty_median)
print(f"\nПропуски в 'кількість' заповнено медіаною: {qty_median}")

print("\n--- Завдання 3: Дублікати ---")
print("Повні дублікати (без subset):", df.duplicated().sum())
print("Дублікати за subset=['id']:", df.duplicated(subset=["id"]).sum())

df = df.drop_duplicates(subset=["id"])
print(f"Залишилось рядків після видалення дублікатів: {len(df)}")

print("\n--- Завдання 4: Типи даних та категорії ---")
df["ціна"] = df["ціна"].astype(str).str.replace(" грн", "").astype(float)

df["місто"] = df["місто"].str.strip().str.capitalize()

print("Унікальні значення міста після чищення:", df["місто"].unique())
print("Типи даних:")
print(df.dtypes)

print("\n--- Завдання 5: Викиди (IQR) ---")
q1 = df["ціна"].quantile(0.25)
q3 = df["ціна"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(df["ціна"] < lower_bound) | (df["ціна"] > upper_bound)]
print(f"Межі IQR для ціни: [{lower_bound:.2f}, {upper_bound:.2f}]")
print("Знайдені викиди:")
print(outliers)

print("\n--- Фінальний очищений DataFrame ---")
print(df)

--- Завдання 1: Початковий брудний набір ---
   id         місто     ціна  кількість
0   1      Чернігів      690        4.0
1   2    Чернігів    690 грн        NaN
2   3      ЧЕРНІГІВ      690        5.0
3   3      ЧЕРНІГІВ      690        5.0
4   5      Чернігів     6210        4.0
5   6      Чернігів  690 грн        NaN
6   7      Чернігів      690        3.0
7   8      Чернігів      690        6.0

--- Завдання 2: Пропуски ---
Кількість пропусків у кожному стовпці:
id           0
місто        0
ціна         0
кількість    2
dtype: int64

Пропуски в 'кількість' заповнено медіаною: 4.5

--- Завдання 3: Дублікати ---
Повні дублікати (без subset): 1
Дублікати за subset=['id']: 1
Залишилось рядків після видалення дублікатів: 7

--- Завдання 4: Типи даних та категорії ---
Унікальні значення міста після чищення: <StringArray>
['Чернігів']
Length: 1, dtype: str
Типи даних:
id             int64
місто            str
ціна         float64
кількість    float64
dtype: object

--- Завдання 5: Вик

## Обґрунтування та письмові відповіді

### 1. Обґрунтування заповнення пропусків медіаною (Завдання 2)

Для малих вибірок (всього 8 спостережень) середнє арифметичне значення є надзвичайно чутливим до наявності аномальних значень та асиметрії в даних.

**Медіана є робастною (стійкою) статистикою**, оскільки:
* Вона відповідає центральному елементу в упорядкованому ряду значень.
* На відміну від середнього, медіана не зміщується під впливом поодиноких екстремальних величин чи помилок вводу.
* Для невеликої кількості спостережень медіана краще відображає типове ("характерне") значення кількості замовленого товару.

---

### 2. Аналіз та рішення щодо викиду у ціні (Завдання 5)

За результатами розрахунку міжквартильного розмаху (**IQR**):
* **Верхня межа IQR:** $850.0$ грн
* **Знайдений викид:** $7650$ грн (що у 9 разів перевищує базову ціну $850$ грн)

#### Обране рішення
Вважати дане значення **помилкою введення даних** (наприклад, технічний збій, механічна опечатка оператора або внесення загальної вартості чека замість ціни за одиницю товару).

#### Обґрунтування
1. **Предметна область:** За умовами задачі інтернет-магазин продає **один конкретний товар** за зафіксованою роздрібною ціною ($850$ грн).
2. **Неможливість варіації:** Одиниця того самого товару в рамках одного регіонального магазину не може стандартно продаватися в 9 разів дорожче.
3. **Подальші дії:** Таке значення підлягає виправленню (заміні на базову ціну $850.0$ грн) або вилученню з аналізу, оскільки воно не відображає реальну цінову політику.

---

## Відповіді на контрольні питання

### 1. Чому заповнення пропуску середнім (а не медіаною) може спотворити стандартне відхилення?

Заповнення пропусків середнім значенням штучно концентрує дані навколо центру і зазвичай применшує реальну дисперсію. Проте, якщо у вибірці наявний хоча б один неусунений викид (як ціна $7650$ грн), розрахунок самого середнього буде сильно зміщений у бік цього викиду.

Підставивши таке "спотворене" середнє у місця пропусків, ми:
1. Змістимо центр розподілу.
2. Штучно створимо нові викривлені значення, що збільшать відхилення більшості нормальних точок від нового "центра".
3. Отримаємо суттєво спотворене показник **стандартного відхилення ($\sigma$)** у наступних обчисленнях.

---

### 2. Яка різниця між `duplicated()` без `subset` і з `subset` — чому результат може відрізнятись?

* **`duplicated()` без `subset`:** Перевіряє **повний збіг усіх стовпців** у рядку. Повертає `True` лише тоді, коли кожен атрибут запису ідентичний іншому.
* **`duplicated(subset=['...'])`:** Перевіряє унікальність **лише за вказаним переліком стовпців** (наприклад, за унікальним ідентифікатором `id`).

#### Чому результат відрізняється?
Якщо у базі даних є два записи з однаковим `id=3`, але в одному з них місто записано як `"КИЇВ"`, а в іншому як `"Київ"`, то:
* Без `subset`: поверне `False` (оскільки текстові рядки не збігаються повністю).
* З `subset=['id']`: поверне `True` (оскільки ідентифікатор замовлення дублюється).

---

### 3. Чому автоматичне видалення всіх значень поза межами IQR не завжди правильне рішення?

Метод $1.5 \times \text{IQR}$ є лише **математичним/статистичним фільтром**, який не враховує контекст бізнес-домену чи фізичну природу даних:

1. **Реальні аномалії проти помилок:** Значення поза межами IQR можуть бути не помилками вводу, а рідкісними, але реальними подіями (наприклад, велика оптова закупівля, сплеск продажів у Чорну п'ятницю, аномальний показник у медицині).
2. **Втрата важливих сигналів:** Автоматично видаляючи такі дані, ми втрачаємо інформацію про крайні сценарії (подібних до "хвостів" розподілу), які часто є найцікавішими для аналізу ризиків чи бізнес-планування.
3. **Зміщення вибірки:** Бездумне видалення "викидів" штучно звужує природну варіативність даних.